# Prática 2 — Fine-Tuning em CIFAR-10
**ENG4502 — Introdução à Ciência de Dados · PUC-Rio**

Neste notebook você vai avançar além da Feature Extraction (Prática 1) e implementar o **Fine-Tuning Parcial** — uma estratégia mais poderosa que "descongela" seletivamente partes do backbone pré-treinado.

**O que é Fine-Tuning?**  
Na Feature Extraction, o backbone é completamente congelado e apenas a camada final aprende. No Fine-Tuning, descongelamos parte ou toda a rede e continuamos o treinamento com uma taxa de aprendizado muito baixa, permitindo que os pesos pré-treinados se **ajustem** aos padrões específicos da nova tarefa — sem perder o conhecimento geral aprendido no ImageNet.

**Por que "Parcial"?**  
Descongelar tudo de uma vez com uma taxa de aprendizado alta pode causar *catastrophic forgetting* — o modelo esquece o que aprendeu. Descongelar apenas as camadas finais (as mais especializadas e próximas à saída) equilibra adaptação e preservação do conhecimento.

**O que você vai implementar:**
1. Repetir a Feature Extraction como linha de base comparativa.
2. Descongelar seletivamente o bloco `layer4` da ResNet-18 (último bloco residual).
3. Usar **Discriminative Learning Rates** — taxas diferentes para camadas diferentes.
4. Comparar acurácia, loss e erros por classe entre as duas abordagens.

> Complete as seções marcadas com `### SEU CÓDIGO AQUI ###`.

## ▶️ Passo 0 — Ativar a GPU (importante!)

No menu do Colab: **Ambiente de execução → Alterar o tipo de ambiente de execução → Acelerador de hardware: GPU (T4)**.

Execute a célula abaixo para confirmar. Neste notebook há dois modelos treinados em sequência — sem GPU cada experimento leva ~5–10 min por época, totalizando até 1h30 no CPU.

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'✅ GPU ativa: {torch.cuda.get_device_name(0)}')
else:
    device = torch.device('cpu')
    print('⚠️  GPU NÃO detectada — ative em: Ambiente de execução → Alterar o tipo → GPU (T4)')
    print('   Sem GPU, dois experimentos × 5 épocas ≈ até 1h30 no total.')

## 0. Imports e Configurações

Além dos imports da Prática 1, incluímos:
- **`sklearn.metrics`**: `confusion_matrix` e `ConfusionMatrixDisplay` para visualizar os erros por classe — já vem instalado no Colab.
- **`CLASSES`**: lista com os nomes das 10 classes do CIFAR-10, usada nos eixos das matrizes de confusão.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import Subset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import time

CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando o dispositivo: {device}')

## 1. Carregamento dos Dados com Subset Balanceado

Mesma configuração da Prática 1: **5.000 imagens de treino** (500 por classe) e **1.000 de validação** (100 por classe), redimensionadas para 224×224 com normalização ImageNet (`mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`).

Os DataLoaders já estão configurados com `num_workers=0` (compatível com Colab e Windows) e `batch_size=64`. O subset é extraído lendo apenas os rótulos (`dataset.targets`), sem carregar imagens na memória desnecessariamente.

In [ ]:
N_TRAIN = 5000
N_VAL = 1000

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_full = datasets.CIFAR10(root='data', train=True, download=True, transform=transform)
val_full = datasets.CIFAR10(root='data', train=False, download=True, transform=transform)

def extract_balanced_subset(dataset, n_total):
    if n_total is None:
        return dataset
    n_per_class = n_total // 10
    indices = []
    class_counts = {c: 0 for c in range(10)}
    for idx, label in enumerate(dataset.targets):
        if class_counts[label] < n_per_class:
            indices.append(idx)
            class_counts[label] += 1
        if len(indices) == n_total:
            break
    return Subset(dataset, indices)

train_dataset = extract_balanced_subset(train_full, N_TRAIN)
val_dataset = extract_balanced_subset(val_full, N_VAL)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
print(f'Treino: {len(train_dataset)} imagens | Validação: {len(val_dataset)} imagens')

## 2. Funções Gerais de Loops

Funções reutilizadas nos dois experimentos (Feature Extraction e Fine-Tuning):

**`train_epoch`:** executa uma época de treino completa. Para cada batch, faz forward pass, calcula a loss, propaga gradientes (`backward`) e atualiza os pesos (`step`). Retorna a loss média ponderada pelo tamanho dos batches.

**`evaluate`:** calcula a acurácia no conjunto de validação usando `torch.no_grad()` (sem cálculo de gradientes, mais rápido e eficiente em memória). Retorna também as listas completas de predições (`all_preds`) e rótulos reais (`all_labels`) — necessárias para gerar a matriz de confusão.

**`train_model`:** encapsula o loop de épocas completo. Chama `train_epoch` e `evaluate` a cada época, registra o histórico de loss e acurácia e imprime o progresso. Retorna o `history` para plotar as curvas de aprendizado ao final.

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    return running_loss / len(loader.dataset)

def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    return acc, all_preds, all_labels

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=5, label=''):
    history = {'loss': [], 'acc': []}
    for epoch in range(num_epochs):
        t0 = time.time()
        loss = train_epoch(model, train_loader, criterion, optimizer, device)
        acc, _, _ = evaluate(model, val_loader, device)
        elapsed = time.time() - t0
        history['loss'].append(loss)
        history['acc'].append(acc)
        print(f'[{label}] Época {epoch+1}/{num_epochs} | Loss: {loss:.4f} | Val Acc: {acc*100:.2f}% | {elapsed:.1f}s')
    return history

## 3. Experimento A — Feature Extraction (Linha de Base)

Repetimos a Feature Extraction da Prática 1 para ter um ponto de comparação justo: **mesmos dados, mesma arquitetura, mesma quantidade de épocas**. A única diferença entre o Experimento A e o B será o número de parâmetros treinados e as taxas de aprendizado.

- Todos os parâmetros do backbone ficam **congelados** (`requires_grad=False`).
- Apenas a nova camada `fc` (512→10) é treinada, com `lr=0.01`.
- **Parâmetros treináveis:** 5.130 (apenas a `fc`).
- **Benchmark esperado:** ~70–75% de acurácia após 5 épocas.

In [ ]:
model_fe = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for p in model_fe.parameters():
    p.requires_grad = False

model_fe.fc = nn.Linear(model_fe.fc.in_features, 10)
model_fe = model_fe.to(device)

optimizer_fe = optim.SGD(model_fe.fc.parameters(), lr=0.01, momentum=0.9)
criterion = nn.CrossEntropyLoss()

print('=== Treinamento: Feature Extraction ===')
history_fe = train_model(model_fe, train_loader, val_loader, criterion, optimizer_fe, num_epochs=5, label='FE')

## 4. Experimento B — Fine-Tuning Parcial com Taxas Discriminativas

**Descongelamento Seletivo (Exercício 1):**  
A ResNet-18 é dividida em blocos: `layer1` → `layer2` → `layer3` → `layer4` → `fc`. As primeiras camadas aprendem features genéricas (bordas, texturas) úteis para qualquer tarefa visual. As camadas finais aprendem representações mais específicas do domínio. Descongelar apenas `layer4` permite que o modelo refine as features de alto nível para o CIFAR-10 sem destruir as representações fundamentais das primeiras camadas.

**Discriminative Learning Rates (Exercício 2):**  
Por que taxas diferentes para grupos de camadas diferentes?
- Os pesos do `layer4` já têm um bom ponto de partida (pré-treinados no ImageNet) — uma taxa alta os destruiria (*catastrophic forgetting*). Usamos `lr = 1e-4` para um **refinamento suave**.
- Os pesos do `fc` são aleatórios e precisam convergir mais rápido. Usamos `lr = 1e-3` para **aprendizado mais agressivo**.

**Parâmetros treináveis:** `layer4` (~2.6M) + `fc` (5.130) ≈ **2.6M parâmetros** — muito mais do que a Feature Extraction, portanto cada época será mais lenta.

In [ ]:
model_ft = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# ==========================================
# EXERCÍCIO 1: Descongelamento Seletivo
# 1. Congele todos os pesos de model_ft inicialmente (requires_grad = False)
# 2. Descongele APENAS os parâmetros pertencentes a model_ft.layer4
# ==========================================
# 💡 Dica Ex.1: for param in model_ft.layer4.parameters(): param.requires_grad = True
### SEU CÓDIGO AQUI ###


# Substituir a camada final fc (pesos aleatórios para as 10 classes do CIFAR-10)
model_ft.fc = nn.Linear(model_ft.fc.in_features, 10)
model_ft = model_ft.to(device)

# ==========================================
# EXERCÍCIO 2: Taxas de Aprendizado Discriminativas
# Configure o otimizador SGD com dois grupos de parâmetros:
# - lr = 1e-4 para model_ft.layer4 (refinamento suave — pesos pré-treinados)
# - lr = 1e-3 para model_ft.fc    (aprendizado rápido — pesos aleatórios)
# Use momentum=0.9
# ==========================================
# 💡 Dica Ex.2:
# optimizer_ft = optim.SGD(
#     [{'params': model_ft.layer4.parameters(), 'lr': 1e-4},
#      {'params': model_ft.fc.parameters(),    'lr': 1e-3}], momentum=0.9)
### SEU CÓDIGO AQUI ###
optimizer_ft = None

trainable_ft = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
print(f'Parâmetros Treináveis (Fine-Tuning Parcial): {trainable_ft:,}')

print('\n=== Treinamento: Fine-Tuning Parcial ===')
history_ft = train_model(model_ft, train_loader, val_loader, criterion, optimizer_ft, num_epochs=5, label='FT')

## 5. Análise Comparativa — Curvas de Aprendizado

Os dois gráficos lado a lado mostram:
- **Esquerda (Acurácia):** como cada estratégia evolui ao longo das épocas. O Fine-Tuning deve superar a Feature Extraction, especialmente nas épocas finais, pois mais parâmetros se adaptam à tarefa.
- **Direita (Loss de Treino):** a loss do Fine-Tuning tende a cair mais rapidamente, pois mais gradientes estão fluindo pela rede.

> Se a curva de acurácia do Fine-Tuning ficar **abaixo** da Feature Extraction, isso indica que a taxa de aprendizado do `layer4` está alta demais — os pesos pré-treinados estão sendo destruídos antes de se adaptar.

In [ ]:
epochs = range(1, 6)
plt.figure(figsize=(12, 5))

# Gráfico de Acurácia
plt.subplot(1, 2, 1)
plt.plot(epochs, [a*100 for a in history_fe['acc']], 'o-', color='#1f77b4', label='Feature Extraction')
plt.plot(epochs, [a*100 for a in history_ft['acc']], 's-', color='#ff7f0e', label='Fine-Tuning Parcial')
plt.title('Acurácia de Validação por Época')
plt.xlabel('Época')
plt.ylabel('Acurácia (%)')
plt.grid(True, alpha=0.3)
plt.legend()

# Gráfico de Loss
plt.subplot(1, 2, 2)
plt.plot(epochs, history_fe['loss'], 'o-', color='#1f77b4', label='Feature Extraction')
plt.plot(epochs, history_ft['loss'], 's-', color='#ff7f0e', label='Fine-Tuning Parcial')
plt.title('Loss de Treinamento por Época')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

## 6. Matriz de Confusão Comparativa

A matriz de confusão mostra **onde cada modelo erra**: cada linha representa a classe real, cada coluna representa a classe predita. A diagonal principal (células mais escuras) são os acertos; os elementos fora da diagonal revelam os pares de classes mais frequentemente confundidos.

O que observar:
- O modelo confunde `cat` com `dog`? `automobile` com `truck`? Essas confusões fazem sentido visualmente — são classes com características visuais parecidas.
- O Fine-Tuning reduz as confusões mais frequentes da Feature Extraction, ou distribui os erros de forma diferente?
- Alguma classe tem desempenho muito pior do que as outras? Isso pode indicar que o backbone pré-treinado tem features menos úteis para aquela categoria.

In [ ]:
_, preds_fe, labels_fe = evaluate(model_fe, val_loader, device)
_, preds_ft, labels_ft = evaluate(model_ft, val_loader, device)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

cm_fe = confusion_matrix(labels_fe, preds_fe)
ConfusionMatrixDisplay(confusion_matrix=cm_fe, display_labels=CLASSES).plot(ax=ax1, cmap='Blues', colorbar=False)
ax1.set_title('Feature Extraction')
plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')

cm_ft = confusion_matrix(labels_ft, preds_ft)
ConfusionMatrixDisplay(confusion_matrix=cm_ft, display_labels=CLASSES).plot(ax=ax2, cmap='Oranges', colorbar=False)
ax2.set_title('Fine-Tuning Parcial (layer4)')
plt.setp(ax2.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 7. Discussão dos Resultados

Com base nos gráficos e matrizes de confusão que você gerou, responda:

1. **Como o Fine-Tuning Parcial se compara com a Feature Extraction** em termos de acurácia final? A diferença foi significativa? O custo computacional extra (mais tempo por época, mais parâmetros) valeu a pena?

2. **Por que é fundamental usar um learning rate muito menor no bloco `layer4`** do que na camada `fc` final? O que aconteceria concretamente se usássemos `lr = 0.01` para ambos?